# 모듈 3: 손실 함수(Loss Function) 실습

이 노트북은 교안(`module_3_loss_function.md`) 흐름에 맞춰
**MSE → BCE(숫자 대입) → CE(숫자 대입) → 공개 데이터셋 적용** 순서로 진행합니다.

목표는 "수식이 실제 코드에서 어떻게 계산되는지"를 공개 데이터셋으로 직접 확인하는 것입니다.

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

## 1) MSE: 회귀 문제의 기본 손실

공식:
$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

예측값과 정답값의 차이를 제곱하고 평균냅니다.

In [ ]:
# 예측값과 정답값(회귀)
pred = torch.tensor([2.5, 5.0, 7.5])
target = torch.tensor([3.0, 5.0, 7.0])

# A. PyTorch 내장 MSE
mse_builtin = nn.MSELoss()(pred, target)

# B. 수식 기반 수작업 MSE
mse_manual = torch.mean((pred - target) ** 2)

print(f"PyTorch MSE: {mse_builtin.item():.4f}")
print(f"수작업 MSE: {mse_manual.item():.4f}")
print(f"각 샘플의 제곱오차: {((pred - target) ** 2).tolist()}")

## 2) BCE: 이진 분류 손실 (숫자 대입)

공식:
$$
BCE = -\left[y\log(\hat{y}) + (1-y)\log(1-\hat{y})\right]
$$

샘플 1개로 보면, 정답이 $y=1$일 때 식은 $-\log(\hat{y})$로 단순화됩니다.
그래서 정답 확률을 높게 줄수록 손실이 작아집니다.

In [ ]:
# 단일 샘플: y=1일 때 BCE가 -log(y_hat)로 줄어드는지 확인
y = torch.tensor([1.0])

yhat_good = torch.tensor([0.9])  # 정답에 가깝게 예측
yhat_bad = torch.tensor([0.1])   # 정답인데 낮은 확률

# 수작업 계산
bce_good_manual = -(y * torch.log(yhat_good) + (1 - y) * torch.log(1 - yhat_good)).mean()
bce_bad_manual = -(y * torch.log(yhat_bad) + (1 - y) * torch.log(1 - yhat_bad)).mean()

# PyTorch 내장 BCE
bce_good_builtin = nn.BCELoss()(yhat_good, y)
bce_bad_builtin = nn.BCELoss()(yhat_bad, y)

print(f"good 예측(0.9) - 수작업: {bce_good_manual.item():.4f}, 내장: {bce_good_builtin.item():.4f}")
print(f"bad  예측(0.1) - 수작업: {bce_bad_manual.item():.4f}, 내장: {bce_bad_builtin.item():.4f}")
print("해석: 정답인데 확률을 낮게 주면(0.1) 손실이 크게 증가합니다.")

### 2-1) BCE 배치 계산도 동일한지 확인

실전에서는 샘플 여러 개를 평균내어 손실을 계산합니다.

In [ ]:
prob_pred = torch.tensor([0.9, 0.2, 0.4])
label = torch.tensor([1.0, 0.0, 1.0])

# A. 내장 BCE
bce_batch_builtin = nn.BCELoss()(prob_pred, label)

# B. 수작업 BCE
bce_batch_manual = -torch.mean(label * torch.log(prob_pred) + (1 - label) * torch.log(1 - prob_pred))

print(f"배치 BCE - PyTorch: {bce_batch_builtin.item():.4f}")
print(f"배치 BCE - 수작업: {bce_batch_manual.item():.4f}")

## 3) CE: 다중 분류 손실 (숫자 대입)

공식:
$$
CE = -\sum_{i=1}^{m} y_i\log(\hat{y}_i)
$$

정답을 one-hot으로 두면(예: `[0,1,0]`) 정답 클래스 항만 남아
결국 CE는 `-log(정답 클래스 확률)`이 됩니다.

In [ ]:
# 클래스 3개: [고양이, 강아지, 토끼], 정답은 강아지(index=1)
target_index = torch.tensor([1])
one_hot = torch.tensor([0.0, 1.0, 0.0])

# 좋은 예측과 나쁜 예측 확률
p_good = torch.tensor([0.2, 0.7, 0.1])
p_bad = torch.tensor([0.7, 0.2, 0.1])

# 수작업 CE: -sum(y*log(p))
ce_good_manual = -(one_hot * torch.log(p_good)).sum()
ce_bad_manual = -(one_hot * torch.log(p_bad)).sum()

# PyTorch CrossEntropyLoss는 logits 입력을 받으므로 log(p)를 logits로 사용
ce_loss = nn.CrossEntropyLoss()
ce_good_builtin = ce_loss(torch.log(p_good).unsqueeze(0), target_index)
ce_bad_builtin = ce_loss(torch.log(p_bad).unsqueeze(0), target_index)

print(f"좋은 예측 CE - 수작업: {ce_good_manual.item():.4f}, 내장: {ce_good_builtin.item():.4f}")
print(f"나쁜 예측 CE - 수작업: {ce_bad_manual.item():.4f}, 내장: {ce_bad_builtin.item():.4f}")
print("해석: 정답 클래스 확률(0.7 -> 0.2)이 낮아지면 손실이 커집니다.")

## 4) 공개 데이터셋으로 손실 함수 실제 적용

이번에는 장난감 숫자가 아니라 공개 데이터셋으로 MSE/BCE/CE를 직접 최적화합니다.
데이터셋은 scikit-learn의 공개 벤치마크(`Diabetes`, `Breast Cancer`, `Iris`)를 사용합니다.

In [ ]:
# 4-1) MSE 실제 적용: Diabetes (회귀)
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

reg_model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=0.01)

for _ in range(300):
    optimizer.zero_grad()
    pred = reg_model(X_train_t)
    loss = mse_loss(pred, y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_pred = reg_model(X_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

print(f"Diabetes 테스트 MSE: {test_mse.item():.4f}")

In [ ]:
# 4-2) BCE 실제 적용: Breast Cancer (이진 분류)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

bin_model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid()
)
bce_loss = nn.BCELoss()
optimizer = torch.optim.Adam(bin_model.parameters(), lr=0.01)

for _ in range(250):
    optimizer.zero_grad()
    prob = bin_model(X_train_t)
    loss = bce_loss(prob, y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_prob = bin_model(X_test_t)
    test_bce = bce_loss(test_prob, y_test_t)
    test_pred = (test_prob >= 0.5).float()
    test_acc = (test_pred == y_test_t).float().mean()

print(f"Breast Cancer 테스트 BCE: {test_bce.item():.4f}")
print(f"Breast Cancer 테스트 정확도: {test_acc.item():.4f}")

In [ ]:
# 4-3) CE 실제 적용: Iris (다중 분류)
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

multi_model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)
ce_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(multi_model.parameters(), lr=0.01)

for _ in range(250):
    optimizer.zero_grad()
    logits = multi_model(X_train_t)
    loss = ce_loss(logits, y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_logits = multi_model(X_test_t)
    test_ce = ce_loss(test_logits, y_test_t)
    test_pred = torch.argmax(test_logits, dim=1)
    test_acc = (test_pred == y_test_t).float().mean()

print(f"Iris 테스트 CE: {test_ce.item():.4f}")
print(f"Iris 테스트 정확도: {test_acc.item():.4f}")